# YOLO11x Training - Chateau Combo Card Detection

Ce notebook entraîne un modèle **YOLO11x** (le plus gros) pour détecter et identifier les 92 cartes du jeu Château Combo.

**Configuration :**
- Modèle : YOLO11x (57M params, +2% mAP vs 11l)
- Résolution : 1280px
- Batch : 8, Workers : 8
- Epochs : 50 avec early stopping

**Prérequis :** Dataset uploadé sur Google Drive dans `My Drive/chato-combourg/dataset/`

**Durée estimée :** ~2-3 heures

## 1. Configuration GPU

In [ ]:
# Vérifier le GPU disponible
!nvidia-smi

## 2. Installation des dépendances

In [ ]:
!pip install ultralytics -q

## 3. Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Vérifier que le dataset existe
import os

DRIVE_PATH = "/content/drive/MyDrive/chato-combourg"
DATASET_PATH = f"{DRIVE_PATH}/dataset"

if os.path.exists(DATASET_PATH):
    print(f"Dataset trouvé: {DATASET_PATH}")
    print(f"Images train: {len(os.listdir(f'{DATASET_PATH}/images/train'))}")
    print(f"Images val: {len(os.listdir(f'{DATASET_PATH}/images/val'))}")
else:
    print(f"ERREUR: Dataset non trouvé à {DATASET_PATH}")
    print("Uploadez le dossier 'dataset' dans Google Drive > chato-combourg/")

## 4. Configuration du dataset

In [ ]:
# Créer le fichier data.yaml avec le bon chemin
data_yaml = f"""# Chateau Combo Card Detection Dataset
path: {DATASET_PATH}
train: images/train
val: images/val

nc: 92

names:
  0: Son Altesse
  1: Imprimeuse
  2: Duchesse
  3: Conspirateur
  4: Pélerin
  5: Aumônier
  6: Maître de guilde
  7: Souffleur de verre
  8: Garde royal
  9: Dame au masque de fer
  10: Professeur
  11: Châtelaine
  12: Prince
  13: Intendant
  14: Dramaturge
  15: Juge
  16: Templier
  17: Sa Majesté la reine
  18: Bouffon
  19: Banquière
  20: Astronome
  21: Officier
  22: Chevaleresse
  23: Architecte
  24: Doyenne
  25: Baron
  26: Générale
  27: Princesse
  28: Veilleur
  29: Orfèvre
  30: Capitaine
  31: Alchimiste
  32: Apothicaire
  33: Flagorneur
  34: La main du Cardinal
  35: Fossoyeur
  36: Nonne
  37: Sa Sainteté
  38: Mécène
  39: Prêteur sur gages
  40: Maître d'armes
  41: Scribe
  42: Milicien
  43: Dévot
  44: Artificier
  45: Chancelière
  46: Mère supérieure
  47: Cardinale
  48: Bûcheron
  49: Miraculée
  50: Curé
  51: Espion
  52: Apiculteur
  53: Mercenaire
  54: Bâtard
  55: Roi des gueux
  56: Forgeronne
  57: Aubergiste
  58: Potier
  59: Horlogère
  60: Sculptrice
  61: Mendiante
  62: Agricultrice
  63: Sorcière
  64: Armurière
  65: Serrurier
  66: Bergère
  67: Médecin
  68: Brigand
  69: Prince des voleurs
  70: Épicière
  71: Barbare
  72: Tailleuse de pierre
  73: Usurpateur
  74: Faussaire
  75: Vigneron
  76: Colporteur
  77: Inventeur
  78: Tire-laine
  79: Écuyer
  80: Fermière
  81: Charpentier
  82: Philosophe
  83: Bourreau
  84: Boulangère
  85: Voyageuse
  86: Pêcheur
  87: Voyante
  88: Village
  89: Château
  90: Révolutionnaire
  91: Moine
"""

with open("/content/data.yaml", "w") as f:
    f.write(data_yaml)

print("data.yaml créé avec succès")

## 5. Entraînement YOLO11

In [ ]:
from ultralytics import YOLO

# Charger le modèle le plus performant
model = YOLO("yolo11x.pt")  # 57M params, meilleur mAP

# Lancer l'entraînement (optimisé H100)
results = model.train(
    data="/content/data.yaml",
    epochs=50,        # Plus d'epochs pour meilleure convergence
    imgsz=1280,        # Résolution haute pour meilleure précision
    batch=8,
    device=0,
    workers=8,
    
    # Hyperparamètres
    lr0=0.001,
    lrf=0.01,
    optimizer="AdamW",
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,
    amp=True,
    
    # Augmentations
    mosaic=0.5,
    mixup=0.05,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,
    degrees=5.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.0,        # Pas de flip (cartes ont un sens)
    flipud=0.0,
    
    # Callbacks
    patience=5,       # Early stopping si pas d'amélioration
    plots=True,
    save=True,
    project=f"{DRIVE_PATH}/runs",
    name="chateau_combo_x",
    exist_ok=True,
)

## 6. Validation

In [ ]:
# Valider le modèle
metrics = model.val()

print(f"\nmAP50-95: {metrics.box.map:.4f}")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP75: {metrics.box.map75:.4f}")

## 7. Export ONNX

In [ ]:
# Exporter en ONNX
model.export(format="onnx", dynamic=True)
print(f"\nModèle exporté dans: {DRIVE_PATH}/runs/chateau_combo_x/weights/")

## 8. Télécharger le modèle

In [ ]:
# Le modèle est déjà sauvegardé sur Google Drive
print("Modèles disponibles sur Google Drive:")
print(f"  - {DRIVE_PATH}/runs/chateau_combo_x/weights/best.pt")
print(f"  - {DRIVE_PATH}/runs/chateau_combo_x/weights/best.onnx")
print(f"  - {DRIVE_PATH}/runs/chateau_combo_x/weights/last.pt")
print(f"\nGraphiques et métriques dans:")
print(f"  - {DRIVE_PATH}/runs/chateau_combo_x/")